<a href="https://colab.research.google.com/github/MACHARODRIGO/tRNASec-Study-Project/blob/main/4_tRNA_ml_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ????? tRNA Machine Learning Exploration  

### ?? Objective  
This notebook explores how **basic sequence-derived features** can be used to classify **tRNAs** by their domain of life (**????? Eukarya vs ?? Bacteria**).  

---

### ? Goals  
- ?? Parse a **multifasta** file of tRNA sequences and link them to their species and domains.  
- ?? Extract simple **sequence features**:  
  - ?? Nucleotide composition (**%A, %U, %G, %C**)  
  - ?? Motif presence (**anticodon UCA**, **codon UGA**)  
  - ?? k-mer frequencies (**2-mers**)  
  - ?? Sequence length  
- ??? Train baseline **ML classifiers**: ?? Decision Tree, ?? Random Forest, ?? SVM.  
- ?? Evaluate models with:  
  - ?? Cross-validation  
  - ?? Test set metrics  
  - ? Feature importance  
  - ?? Learning curves  

---

### ?? Limitations  
- Current feature set is **simplistic** (composition + motifs), without:  
  - ??? Structural descriptors  
  - ?? Modification data  
- Predictions are restricted to a **binary classification**: ?? Bacteria vs ????? Eukarya.  
- Goal = **prototype an ML workflow** ????? rather than biological novelty.  


In [ ]:
!pip install biopython gdown


In [ ]:
import gdown
from Bio import SeqIO
import pandas as pd
import numpy as np
from itertools import product
import requests
import os
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
def parse_fasta_with_species(fasta_file):
    records = []
    for rec in SeqIO.parse(fasta_file, "fasta"):
        desc = rec.description
        seq = str(rec.seq).upper().replace("T", "U")

        # ID limpio (solo URSxxxxx)
        clean_id = desc.split("|")[0]   # todo lo que est? antes del primer "|"

        # especie: tomo la segunda parte despu?s de "|", reemplazo espacios por "_"
        try:
            species = desc.split("|")[1].replace(" ", "_")
        except IndexError:
            species = "Unknown"

        records.append({
            "id": clean_id,
            "species": species,
            "sequence": seq
        })
    return pd.DataFrame(records)

# ----------------------------
# 2. Unir con CSV de dominios
# ----------------------------
def merge_with_domain(fasta_df, csv_file):
    domain_df = pd.read_csv(csv_file)

    # aseguramos consistencia de nombre
    if "Species" in domain_df.columns:
        species_col = "Species"
    elif "species" in domain_df.columns:
        species_col = "species"
    elif "Organism" in domain_df.columns:
        species_col = "Organism"
    else:
        raise ValueError(f"?? No encontr? columna de especie en CSV. Columnas disponibles: {domain_df.columns}")

    # normalizar nombres
    domain_df["species"] = domain_df[species_col].str.replace(" ", "_")
    domain_df["domain"] = domain_df["domain"]
    merged = fasta_df.merge(domain_df, on="species", how="left")
    return merged
# ----------------------------
# 3. Extraer features adicionales
# ----------------------------
def extract_seq_features(df, k=2):
    kmers = ["".join(p) for p in product("AUGC", repeat=k)]
    feature_rows = []

    for _, row in df.iterrows():
        seq = row["sequence"]
        length = len(seq)

        perc = {nt: seq.count(nt)/length for nt in "AUGC"}
        counts = {kmer: seq.count(kmer)/(length-k+1) for kmer in kmers}

        has_UCA = 1 if "UCA" in seq else 0
        has_UGA = 1 if "UGA" in seq else 0

        feat = {
            "id": row["id"],
            "species": row["species"],
            "domain": row["domain"],  # ya viene del CSV
            "length": length,
            **perc,
            **counts,
            "has_UCA": has_UCA,
            "has_UGA": has_UGA
        }
        feature_rows.append(feat)

    return pd.DataFrame(feature_rows)

# ----------------------------
# 4. Pipeline completo
# ----------------------------
def build_feature_table(fasta_file, csv_file, k=2):
    fasta_df = parse_fasta_with_species(fasta_file)
    merged_df = merge_with_domain(fasta_df, csv_file)
    feature_df = extract_seq_features(merged_df, k=k)
    return feature_df

In [ ]:
import gdown

# FASTA
file_id = "14sQs3vaVxI8f5pWaxVZzlBVudj8tXKbY"
url = f"https://drive.google.com/uc?id={file_id}"
fasta_file = gdown.download(url, "sequences.fasta", quiet=False)  # ahora fasta_file tiene el path correcto

# CSV
csv_file_id = "1VLxpNfkPrd1-tbtx6tqu8BDwXDa1oljW"
csv_url = f"https://drive.google.com/uc?id={csv_file_id}"
feature_file = gdown.download(csv_url, "features.csv", quiet=False)  # idem


In [ ]:
# Paso 1: generar tabla completa

final_df = build_feature_table(fasta_file, feature_file, k=2)

# Paso 2: revisar

print(final_df.head())


# Paso 3: guardar para usar en ML
final_df.to_csv("tRNA_features_final.csv", index=False)


#

# ?? Machine Learning Pipeline for tRNA

## Step 1 - Prepare Dataset

In [ ]:
# Drop rows with missing labels
df = final_df.dropna(subset=["domain"])
df = df[df["domain"].isin(["E", "B"])]

# Features (numeric only)
X = df.drop(columns=["id", "species", "domain"])
y = df["domain"]

print("\nFeature matrix shape:", X.shape)
print("Target vector distribution:\n", y.value_counts())

## Step 2 : Train/Test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set size:", X_train.shape[0])
print("Test set size:", X_test.shape[0])

## Step 3:  Train models

### Decision Tree

In [ ]:
tree_clf = DecisionTreeClassifier(max_depth=5, random_state=42)
tree_clf.fit(X_train, y_train)
print("Decision Tree Accuracy:", tree_clf.score(X_test, y_test))


Random Forest

In [ ]:
rf_clf = RandomForestClassifier(n_estimators=200, random_state=42)
rf_clf.fit(X_train, y_train)
print("Random Forest Accuracy:", rf_clf.score(X_test, y_test))

Support Vector Machine

In [ ]:
svm_clf = make_pipeline(StandardScaler(), SVC(kernel="rbf"))
svm_clf.fit(X_train, y_train)
print("SVM Accuracy:", svm_clf.score(X_test, y_test))


## Step 4 : Cross-validation


In [ ]:
print("\nStep 4: Cross-validation (all models, cv=5)...")

for name, model in [
    ("Decision Tree", tree_clf),
    ("Random Forest", rf_clf),
    ("SVM", svm_clf),
]:
    scores = cross_val_score(model, X, y, cv=5)
    print(f"{name} CV scores:", scores)
    print(f"{name} Mean CV Accuracy: {np.mean(scores):.3f}\n")

### ?? Interpretation of Results  

- ?? **Decision Tree** ? simpler, interpretable, ~92% accuracy.  
- ?? **Random Forest** ? very robust, ~97.5% accuracy.  
- ?? **SVM** ? strong generalization, ~96.7% accuracy.  
- ? All three achieve **>90% accuracy** separating bacterial from eukaryotic tRNAs.  

## Step 5: Final Evaluation on Test Set

After cross-validation, we evaluate all models on the independent **test set**.  
We report the **classification report** (precision, recall, F1-score) and the **confusion matrix**.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

for name, model in [
    ("Decision Tree", tree_clf),
    ("Random Forest", rf_clf),
    ("SVM", svm_clf),
]:
    print(f"\n=== {name} ===")
    y_pred = model.predict(X_test)
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    print("Confusion Matrix:")

### ?? Interpretation of Results  

- ? All models (?? Decision Tree, ?? Random Forest, ?? SVM) achieved **perfect classification** (100% accuracy).  
- ?? No misclassifications: all 16 Bacterial tRNAs ? B, all 8 Eukaryotic tRNAs ? E.  
- ?? While this validates the ML pipeline, it also shows the dataset is **easily separable** and highlights the need for more complex features (structure, modifications) for deeper biological insights.  

## Step 6: Feature Importance (Tree-based Models Only)

For tree-based models we can measure feature importance, i.e.
which features (k-mers, sequence properties) contribute most to the classification.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

for name, model in [
    ("Decision Tree", tree_clf),
    ("Random Forest", rf_clf),
]:
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1]

    plt.figure(figsize=(10,6))
    plt.title(f"Feature Importance ({name})")
    plt.bar(range(20), importances[indices[:20]], align="center")
    plt.xticks(range(20), X.columns[indices[:20]], rotation=90)
    plt.tight_layout()
    plt.show()


### ?? Interpretation of Results  

- The **?GG? di-nucleotide frequency** ?? dominated classification.  
- Secondary contributors: %A and AU frequency.  
- This shows the model relies on **low-level compositional bias** rather than biological structure ? effective but oversimplified.  


## Step 7: Learning Curves (Loss Tables)

To analyze how performance changes with training set size,
we compute learning curves for each model and export them as ?loss tables?.

In [ ]:
from sklearn.model_selection import learning_curve

loss_tables = []

for name, model in [
    ("Decision Tree", tree_clf),
    ("Random Forest", rf_clf),
    ("SVM", svm_clf),
]:
    train_sizes, train_scores, test_scores = learning_curve(
        model, X, y, cv=5, scoring="accuracy",
        train_sizes=np.linspace(0.1, 1.0, 5)
    )
    train_mean = np.mean(train_scores, axis=1)
    test_mean = np.mean(test_scores, axis=1)

    loss_table = pd.DataFrame({
        "train_size": train_sizes,
        "train_accuracy": train_mean,
        "val_accuracy": test_mean,
        "model": name
    })
    print(f"\n{name} Loss Table:\n", loss_table)
    loss_tables.append(loss_table)

    plt.figure(figsize=(8,5))
    plt.plot(train_sizes, train_mean, "o-", label="Training accuracy")
    plt.plot(train_sizes, test_mean, "o-", label="Validation accuracy")
    plt.xlabel("Training set size")
    plt.ylabel("Accuracy")
    plt.title(f"Learning Curve ({name})")
    plt.legend()
    plt.grid(True)
    plt.show()

# Export all loss tables
all_loss_tables_df = pd.concat(loss_tables)
all_loss_tables_df.to_csv("loss_tables.csv", index=False)


### ?? Interpretation of Results

- ?? **Training accuracy = 1.0** for all models (they memorize training data).  
- Validation accuracy improves with more samples:  
  - ?? Decision Tree ? up to ~0.92  
  - ?? Random Forest ? up to ~0.97  
  - ?? SVM ? up to ~0.97  
- ?? Curves show high variance with small datasets, then stabilize.  
- ?? SVM produced a warning (class imbalance during CV) ? not model error but dataset artifact.  


## ?? Conclusion  
With only **basic sequence-derived features** ??, classical ML models ??? can already distinguish ????? Eukaryotic vs ?? Bacterial tRNAs with very high accuracy.  

But:  
- The approach is **oversimplified** ??, dominated by compositional bias (GG content).  
- Biological interpretability = limited ??.  
- To gain insights, we need to add richer features:  
  - ??? **Structural descriptors** (stems, loops, pairing fraction).  
  - ?? **Modification presence/absence** (?55, m?A58, etc.).  
  - ?? **Evolutionary / clustering signals** across domains.  

?? This notebook serves as a **proof-of-concept** prototype: parsing ? feature extraction ? ML ? evaluation ?.  
Next step = evolve toward **biologically meaningful insights** by integrating **structure** and **epitranscriptomic modifications**.  